# Gemini API — 같은 일을 다른 모델로

`1_1_openai` 에서 OpenAI 모델에게 말을 걸어 봤습니다.
이번에는 **구글의 Gemini** 에게 똑같은 일을 시킵니다.

놀라운 점은 **코드가 거의 바뀌지 않는다**는 것입니다.
Gemini 가 OpenAI 와 같은 모양의 주소(호환 API)를 열어 두었기 때문입니다.

## 이 노트북의 흐름

| 단계 | 내용 |
|---|---|
| **Step 1~2** | 준비 — 라이브러리, 키, 주소 |
| **Step 3** | Gemini 클라이언트 만들기 — **딱 두 군데만 다르다** |
| **Step 4~5** | 첫 대화 — `1_1` 과 같은 메시지 보내기 |
| **Step 6~7** | 역할 주기 — `1_1` 과 같은 보안 전문가 질문 |
| **Step 8** | 덤 — 구글 전용 라이브러리로 부르는 법 |

> **이 노트북에서 기억할 한 가지**
> `OpenAI()` 에 **키와 주소만 바꿔 넣으면** 다른 회사 모델도 같은 코드로 부를 수 있습니다.

## Step 1. 준비물

`1_1_openai` 와 **완전히 같습니다.** Gemini 를 부르는데도 `OpenAI` 를 가져옵니다.

| 가져오는 것 | 하는 일 |
|---|---|
| `os` | 환경변수를 읽는다 |
| `dotenv` | `.env` 파일을 환경변수로 올려 준다 |
| `OpenAI` | OpenAI 공식 클라이언트 — **Gemini 호출에도 그대로 쓴다** |

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

## Step 2. 키와 주소 확인

이번에는 `.env` 에서 **두 가지**를 읽습니다.

```
GOOGLE_API_KEY=AQ....                                               <- 구글 키
GEMINI_URL=https://generativelanguage.googleapis.com/v1beta/openai/   <- Gemini 의 OpenAI 호환 주소
```

키는 `1_1` 처럼 앞 8자만 찍고, 주소는 비밀이 아니므로 전부 찍습니다.

In [ ]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
gemini_url = os.getenv('GEMINI_URL')

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

print(f"Gemini URL: {gemini_url}")

## Step 3. Gemini 클라이언트 만들기 — 두 군데만 다르다

```python
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
```

`1_1` 과 비교해 보세요.

| | `1_1_openai` | `1_2_gemini` |
|---|---|---|
| 코드 | `OpenAI()` | `OpenAI(api_key=..., base_url=...)` |
| 키 | `OPENAI_API_KEY` 를 **알아서** 찾음 | `GOOGLE_API_KEY` 를 **직접** 넘김 |
| 주소 | OpenAI 기본 주소 | `GEMINI_URL` |

### 왜 직접 넘겨야 하나

`OpenAI()` 는 괄호가 비어 있으면 `OPENAI_API_KEY` 와 OpenAI 주소를 씁니다.
Gemini 는 **이름이 다른 키**와 **다른 주소**를 쓰므로 직접 알려 줘야 합니다.

> `base_url` 은 "요청을 어디로 보낼지"입니다.
> 이것만 바꾸면 같은 라이브러리로 **요청이 OpenAI 대신 구글 서버로** 갑니다.

In [ ]:
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

## Step 4. `1_1` 과 같은 메시지 만들기

`1_1_openai` Step 3 의 "홍길동" 메시지를 그대로 씁니다.
같은 메시지를 보내야 **두 모델의 답을 공평하게 비교**할 수 있습니다.

In [ ]:
message = "안녕하세요, 만나뵈서 반갑습니다. 내 이름은 홍길동입니다."
messages = [{"role": "user", "content": message}]
messages

## Step 5. 첫 호출

```python
response = gemini.chat.completions.create(model="gemini-3.8-flash", messages=messages)
```

`1_1` 의 Step 5 와 비교하면 **바뀐 것은 두 단어**입니다.

| | `1_1_openai` | `1_2_gemini` |
|---|---|---|
| 클라이언트 | `openai` | `gemini` |
| 모델 | `gpt-5-nano` | `gemini-3.8-flash` |
| 답 꺼내기 | `response.choices[0].message.content` | **똑같음** |

> **모델 이름은 바뀝니다.**
> 구글은 오래된 모델을 새 사용자에게 막습니다. 예를 들어 `gemini-2.5-flash` 를 쓰면
> `404 ... no longer available to new users` 오류가 나고, 오류 메시지에 **새 모델 이름**이 적혀 옵니다.
> 그 이름으로 바꾸면 됩니다.

In [ ]:
response = gemini.chat.completions.create(model="gemini-3.8-flash", messages=messages)
response.choices[0].message.content

## Step 6. `system` + `user` — `1_1` 과 같은 보안 전문가 질문

`1_1_openai` Step 6~8 의 프롬프트를 그대로 가져옵니다.

In [ ]:
system_prompt = """
당신은 IT 보안 전문가입니다. 
시스템 보안에 대한 전문 지식을 제공하며 새롭게 발생하는 보안 문제에 대해서도 대처 방법을 제공합니다
"""

user_prompt = """
나는 IT 회사의 보안 책임자입니다.
현재 가장 대두되고 있는 보안 이슈는 무엇이 있나요?
"""

messages = [{"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}]

## Step 7. 역할을 준 상태로 호출

`print()` 로 감싸서 목록과 줄바꿈이 제대로 보이게 합니다.

**`1_1` 에서 `gpt-4.1-mini` 가 한 답과 비교해 보세요.**
같은 질문이라도 모델마다 고르는 이슈, 말투, 길이가 다릅니다.

In [ ]:
response = gemini.chat.completions.create(
    model = "gemini-3.8-flash",
    messages = messages
)
print(response.choices[0].message.content)

### 토큰 사용량도 똑같이 볼 수 있다

`1_1` Step 10 에서 본 `usage` 도 같은 자리에 있습니다.
응답의 **모양까지 OpenAI 와 같기** 때문에 꺼내는 코드도 같습니다.

In [ ]:
print("보낸 토큰:", response.usage.prompt_tokens)
print("받은 토큰:", response.usage.completion_tokens)
print("멈춘 이유:", response.choices[0].finish_reason)

## Step 8. 덤 — 구글 전용 라이브러리 `google-genai`

지금까지는 OpenAI 라이브러리를 빌려 썼습니다. 구글이 직접 만든 라이브러리도 있습니다.

```python
from google import genai
client = genai.Client()   # GOOGLE_API_KEY 를 알아서 찾음
```

| | OpenAI 호환 방식 (Step 3~7) | 구글 전용 `google-genai` |
|---|---|---|
| 장점 | **코드 하나로 여러 회사 모델**을 바꿔 가며 쓴다 | 구글의 최신 기능을 **전부** 쓸 수 있다 |
| 키 | 직접 넘김 | `GOOGLE_API_KEY` 를 알아서 찾음 |
| 질문 넣기 | `messages=[...]` | `contents="..."` |
| 답 꺼내기 | `response.choices[0].message.content` | `response.text` |

> 이 과정에서는 **여러 모델을 같은 코드로 비교**하는 것이 목적이므로
> 주로 OpenAI 호환 방식을 씁니다. 전용 라이브러리는 이런 것도 있다는 정도로 알아 두세요.

In [ ]:
from google import genai

client = genai.Client()
response = client.models.generate_content(
    model = "gemini-3.8-flash",
    contents = "안녕하세요, 만나뵈서 반갑습니다. 내 이름은 홍길동입니다."
)
print(response.text)

---

## 정리

```
OpenAI(api_key=구글키, base_url=Gemini주소)  ->  나머지는 1_1 과 똑같이
```

### 꼭 기억할 것

| | 내용 |
|---|---|
| **키와 주소만 바꾼다** | `api_key` 와 `base_url` 두 개로 다른 회사 모델을 부른다 |
| **나머지 코드는 그대로** | `messages`, `create()`, `choices[0].message.content` 모두 같다 |
| **모델 이름은 바뀐다** | 404 오류가 나면 오류 메시지 속 새 이름으로 바꾼다 |
| **전용 라이브러리도 있다** | `google-genai` — 구글 기능을 전부 쓰고 싶을 때 |

### 직접 해 보기

1. Step 7 의 답을 `1_1` 에서 `gpt-4.1-mini` 가 한 답과 나란히 놓고 비교해 보세요. 무엇이 다른가요?
2. `1_1_2_temperature` 의 `ask()` 함수에서 `openai` 를 `gemini` 로, 모델을 `gemini-3.8-flash` 로 바꿔 보세요. Gemini 에서도 temperature 가 똑같이 동작하나요?
3. Step 5 의 모델을 일부러 `gemini-2.5-flash` 로 바꿔 실행해 보세요. 어떤 오류가 나오나요?

### 다음 노트북

| 노트북 | 내용 |
|---|---|
| `1_3_ollama` | 같은 일을 **내 컴퓨터 안의 모델**로 — 이번엔 키도 필요 없다 |